In [1]:
import numpy as np
import pandas as pd


1. Load a CSV file, identify columns with more than 20% missing values, and drop only those columns.

In [19]:
df = pd.read_csv(r'C:\Users\aryan\Projects\data_engineering\daily-tasks\2026-02-18\practice.csv')
df.head()

,OrderID,CustomerID,CustomerName,Region,Category,Product,OrderDate,Sales,Quantity,Age,DiscountCode,Notes
0,1001,C001,Alice,North,Electronics,Phone,2024-01-05,1200,2,25,NaN,First order
1,1002,C002,Bob,South,Clothing,Jacket,2024-01-06,200,1,17,DISC10,NaN
2,1003,C003,Charlie,East,Electronics,Laptop,2024-01-07,1500,1,34,NaN,Priority
3,1004,C001,Alice,North,Clothing,Shirt,2024-01-08,80,3,25,DISC5,NaN
4,1005,C004,David,West,Furniture,Chair,2024-01-09,300,4,45,NaN,NaN


In [20]:
df.dropna(axis=1, thresh=0.8*len(df), inplace=True)

2. Use groupby with transform() to add a column showing each row's value as a percentage of its group total.

In [18]:
df['group_total'] = df.groupby('Region')['Sales'].transform('sum')
df['RegionSalesPCT'] = (df['Sales'] / df['group_total']) * 100
df.head()

,OrderID,CustomerID,CustomerName,Region,Category,Product,OrderDate,Sales,Quantity,Age,group_total,RegionSalesPCT
0,1001,C001,Alice,North,Electronics,Phone,2024-01-05,1200,2,25,2730,43.956044
1,1002,C002,Bob,South,Clothing,Jacket,2024-01-06,200,1,17,1850,10.810811
2,1003,C003,Charlie,East,Electronics,Laptop,2024-01-07,1500,1,34,2520,59.523810
3,1004,C001,Alice,North,Clothing,Shirt,2024-01-08,80,3,25,2730,2.930403
4,1005,C004,David,West,Furniture,Chair,2024-01-09,300,4,45,2400,12.500000



3. Reshape a DataFrame from wide format to long format using pd.melt().

In [21]:
melted = pd.melt(
    df,
    id_vars=["OrderID", "CustomerID", "Region"],
    value_vars=["Sales", "Quantity"],
    var_name="Metric",
    value_name="Value"
)

print(melted.head())

   OrderID CustomerID Region Metric  Value
0     1001       C001  North  Sales   1200
1     1002       C002  South  Sales    200
2     1003       C003   East  Sales   1500
3     1004       C001  North  Sales     80
4     1005       C004   West  Sales    300



4. Given a DataFrame with a datetime column, extract the day of week and create a new column with it, then find which weekday has the highest average sales.

In [23]:
df["OrderDate"] = pd.to_datetime(df["OrderDate"], errors="coerce")

In [25]:
df['weekday'] = df['OrderDate'].dt.day_name()
weekday_avg = (
    df.groupby('weekday')['Sales']
      .mean()
      .sort_values(ascending=False)
)
print(weekday_avg)

weekday
Monday       1040.000000
Friday       1033.333333
Sunday        875.000000
Tuesday       600.000000
Wednesday     360.000000
Saturday      200.000000
Thursday      125.000000
Name: Sales, dtype: float64


5. Merge two DataFrames with different column names representing the same key using appropriate parameters.


In [27]:
customers = pd.read_csv(r"C:\Users\aryan\Projects\data_engineering\daily-tasks\2026-02-18\customerinfo.csv")

merged = pd.merge(
    df,
    customers,
    left_on="CustomerID",
    right_on="CustID",
    how="left"
)

print(merged.head())

   OrderID CustomerID CustomerName Region     Category Product  OrderDate  \
0     1001       C001        Alice  North  Electronics   Phone 2024-01-05   
1     1002       C002          Bob  South     Clothing  Jacket 2024-01-06   
2     1003       C003      Charlie   East  Electronics  Laptop 2024-01-07   
3     1004       C001        Alice  North     Clothing   Shirt 2024-01-08   
4     1005       C004        David   West    Furniture   Chair 2024-01-09   

   Sales  Quantity  Age   weekday CustID LoyaltyLevel  
0   1200         2   25    Friday   C001         Gold  
1    200         1   17  Saturday   C002       Silver  
2   1500         1   34    Sunday   C003         Gold  
3     80         3   25    Monday   C001         Gold  
4    300         4   45   Tuesday   C004       Bronze  



6. Use pd.cut() to bin an age column into labeled groups (Child, Teen, Adult, Senior).

In [28]:
df["AgeGroup"] = pd.cut(
    df["Age"],
    bins=[0, 12, 19, 59, 120],
    labels=["Child", "Teen", "Adult", "Senior"]
)
df.head()

,OrderID,CustomerID,CustomerName,Region,Category,Product,OrderDate,Sales,Quantity,Age,weekday,AgeGroup
0,1001,C001,Alice,North,Electronics,Phone,2024-01-05,1200,2,25,Friday,Adult
1,1002,C002,Bob,South,Clothing,Jacket,2024-01-06,200,1,17,Saturday,Teen
2,1003,C003,Charlie,East,Electronics,Laptop,2024-01-07,1500,1,34,Sunday,Adult
3,1004,C001,Alice,North,Clothing,Shirt,2024-01-08,80,3,25,Monday,Adult
4,1005,C004,David,West,Furniture,Chair,2024-01-09,300,4,45,Tuesday,Adult


In [31]:
df = df.drop_duplicates(
    subset=["CustomerID", "Product", "OrderDate"]
)
df.head()

,OrderID,CustomerID,CustomerName,Region,Category,Product,OrderDate,Sales,Quantity,Age,weekday,AgeGroup
0,1001,C001,Alice,North,Electronics,Phone,2024-01-05,1200,2,25,Friday,Adult
1,1002,C002,Bob,South,Clothing,Jacket,2024-01-06,200,1,17,Saturday,Teen
2,1003,C003,Charlie,East,Electronics,Laptop,2024-01-07,1500,1,34,Sunday,Adult
3,1004,C001,Alice,North,Clothing,Shirt,2024-01-08,80,3,25,Monday,Adult
4,1005,C004,David,West,Furniture,Chair,2024-01-09,300,4,45,Tuesday,Adult



7. Apply a custom function to a grouped DataFrame using apply() that returns the range (max - min) of a column.

In [32]:
def sales_range(group):
    return group["Sales"].max() - group["Sales"].min()

range_result = df.groupby("Region").apply(sales_range)

print(range_result)

Region
East     1380
North    1120
South     750
West     1900
dtype: int64


C:\Users\aryan\AppData\Local\Temp\ipykernel_4404\1596959164.py:4: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  range_result = df.groupby("Region").apply(sales_range)



8. Convert a long-format DataFrame to a pivot table and then reverse it back using stack().

In [33]:
pivot = df.pivot_table(
    index="Region",
    columns="Category",
    values="Sales",
    aggfunc="sum"
)

print(pivot)

Category  Clothing  Electronics  Furniture
Region                                    
East         120.0       1700.0      700.0
North        330.0       1200.0        NaN
South        350.0       1500.0        NaN
West           NaN          NaN     2400.0



9. Chain at least 4 Pandas operations (filter, groupby, agg, sort) in a single expression and explain each step.

In [34]:
result = (
    df[df["Sales"] > 200]                
      .groupby("Region")                 
      .agg(TotalSales=("Sales", "sum"),
           AvgSales=("Sales", "mean"))   
      .sort_values("TotalSales",         
                   ascending=False)
)

print(result)

        TotalSales  AvgSales
Region                      
West          2300    1150.0
East          2200    1100.0
South         1500     750.0
North         1450     725.0
